In [31]:
from dotenv import load_dotenv,find_dotenv
load_dotenv(find_dotenv(),override=True)

True

In [32]:
!pip install google-genai

In [33]:
from google import genai

In [34]:
import os
client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

In [35]:
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3.1-flash-lite
models/gemini-3-pro-image
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/antigravity-preview-05-2026
models/antigravity-preview-09-2026
models/antigravity-preview-latest
models/deep-research-max-preview-04-2026
models/deep-research-preview-04-2026
models/deep-research-pro-preview-12-2025
models/gemini-embedding-001
models/gemini-embedding-2


In [36]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Explain Artificial Intelligence in simple words."
)
print(response.text)

Imagine teaching a computer to think, learn, and make decisions a bit like a human. That is **Artificial Intelligence (AI)**. 

Normally, computers only do what humans explicitly program them to do step-by-step. If you don't write the code for something, the computer can't do it. 

**AI is different.** Instead of giving the computer a strict set of rules, you give it a **huge amount of examples**, and the computer figures out the rules and patterns all by itself.

### Here is a simple analogy:

* **Regular Computer (Old School):** You show a computer 10,000 pictures of cats and write a complex computer program telling it: *"A cat has pointy ears, whiskers, and a tail."* The computer uses your rules to find cats.
* **Artificial Intelligence (AI):** You show a computer 10,000 pictures of cats and dogs, but **you don't give it any rules**. The computer studies the pictures, notices patterns on its own, and learns to recognize a cat all by itself. When you show it a new picture, it says, "

# Load Documents

In [37]:
from langchain_community.document_loaders import PyPDFLoader

In [38]:
data=PyPDFLoader('Artificial_Intelligence_10_Page_Document.pdf')
docs=data.load()

In [39]:
from langchain_core.prompts import ChatPromptTemplate

In [40]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant."),
    ("human", "{data}")
])



In [41]:
prompt = chat_prompt.invoke({
    "data": docs[0].page_content
}).to_string()

print(type(prompt))


<class 'str'>


In [42]:
response=client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents=prompt
)
print(response.text)

It looks like you've shared the title page and introduction for a 10-page educational document on **Artificial Intelligence**. 

How can I help you with it today? For example, I can:

* **Summarize** the upcoming content if you provide the rest of the pages.
* **Explain specific concepts** related to AI's history, uses, benefits, risks, or future.
* **Generate study questions or a quiz** based on the document.
* **Help you write or expand** upon this topic.

Please feel free to paste the next sections or let me know what you'd like to do!


In [43]:
pip install -U langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [44]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [45]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
texts = text_splitter.split_documents(docs)

In [46]:
print(len(texts))

26


In [47]:
print(texts[0].page_content)

Artificial Intelligence | Educational Document
Page 1
 ARTIFICIAL
INTELLIGENCE
 An Introduction to Its History, Uses, Benefits, Risks, and Future
 A 10-Page Educational Document
 Prepared for general educational and study purposes


In [48]:
!pip install -U langchain-chroma langchain-google-genai

In [49]:
!pip install -U langchain-core langchain-google-genai langchain-chroma langchain-text-splitters langchain-community

In [50]:
import langchain_core
import langchain_google_genai
import langchain_chroma

print("LangChain Core:", langchain_core.__version__)
print("All packages imported successfully!")

LangChain Core: 1.7.6
All packages imported successfully!


In [51]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

print("Imports Successful!")

Imports Successful!


In [52]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)



In [53]:
vector_store = Chroma(
    collection_name="docs",
    embedding_function=embeddings,
    persist_directory="chroma_langchain_db",
)

In [54]:
vector_store.add_documents(documents=texts)

print("PDF documents added successfully!")

PDF documents added successfully!


In [55]:
query = "What is Artificial Intelligence?"

results = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"\nDocument {i+1}:")
    print(doc.page_content)


Document 1:
Artificial Intelligence | Educational Document
Page 2
Introduction to Artificial Intelligence
Overview
Artificial Intelligence (AI) is a branch of computer science concerned with creating machines and software that can
perform tasks commonly associated with human intelligence. These tasks include understanding language,
recognizing images, learning from experience, solving problems, making predictions, and supporting decisions. AI
is not a single machine or program. It is a broad collection of methods, technologies, and applications that allow
computer systems to respond intelligently to information.
Why AI Matters
AI has moved from research laboratories into everyday life. It helps people search the internet, receive
recommendations, translate languages, detect suspicious banking activity, navigate roads, and access digital
assistants. Businesses use AI to analyse large amounts of information and improve services. Governments and

Document 2:
others remain fixed until dev

In [56]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Answer the question using only the following context.
If the answer is not available in the context, say:
"I could not find this information in the document."

Context:
{context}

Question:
{question}

Answer:
""")

In [57]:
query = "What is Artificial Intelligence?"

results = vector_store.similarity_search(query, k=3)

context = "\n\n".join(
    doc.page_content for doc in results
)

prompt = rag_prompt.invoke({
    "context": context,
    "question": query
}).to_string()

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

print(response.text)

Based on the provided document, Artificial Intelligence (AI) is a branch of computer science concerned with creating machines and software that can perform tasks commonly associated with human intelligence. It is not a single machine or program, but a broad collection of methods, technologies, and applications that allow computer systems to respond intelligently to information.
